# Factoid Test Evaluation: Qwen 2.5 0.5B Models

Evaluates one or more local SFT/DPO adapters on the four BioASQ Task 13B golden test batches. It uses the same full-resource Qwen chat prompt, greedy decoding, and factoid MRR calculation as the DPO generated-dev evaluation.

The test set should be used for final reporting, not to choose hyperparameters or checkpoints.


Optional second-pass recovery is configured by `ENABLE_SECOND_PASS_RECOVERY`. The reviewer uses the strict-extractive SFT by default and appends at most one validated answer at rank 2. The original answer remains rank 1. The approved prompt requests a second answer without abstention; malformed or duplicate attempts are retried once, then logged without appending an invalid candidate. Gold answers are used only for scoring. Gold-snippet data variants remain oracle diagnostics.

With recovery enabled, `summary.json` reports ranked MRR alongside `first_pass_mrr` and `recovered_at_rank2_count`. `first_pass_metrics.json` preserves the baseline, and `second_pass_recovery.jsonl` stores reviewer prompts and outputs. Official Java scores and submission files are written under `official_bioasq/`. Restart the kernel before running this revised notebook.


In [ ]:
import gc
import json
import os
import sys
import time
from datetime import datetime
from pathlib import Path

os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
os.environ.setdefault('HF_HUB_OFFLINE', '1')
os.environ.setdefault('TRANSFORMERS_OFFLINE', '1')

import numpy as np
import pandas as pd
import torch
from peft import PeftModel
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents, start / 'Task-Structured Counterfactual Preference Mining']:
        if (candidate / 'cse_dpo').is_dir() and (candidate / 'src').is_dir():
            return candidate.resolve()
    raise RuntimeError('Could not locate the project root. Start the notebook in the project or workspace directory.')


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from cse_dpo.generated_bioasq_eval import build_generated_eval_rows, evaluate_generated_bioasq, load_gold_examples
from cse_dpo.normalize_set_answers import normalize_answer_surface
from src.utility.eval_dataset import build_messages, load_eval_examples, render_prompt
from src.utility.eval_types import EvalExample

SINGLE_ANSWER_PROMPT_PATH = PROJECT_ROOT / 'prompts/factoid_single_answer_aligned.json'
SINGLE_ANSWER_PROMPT_ID = 'factoid-single-answer-aligned-v1'
_prompt_registry = json.loads(SINGLE_ANSWER_PROMPT_PATH.read_text(encoding='utf-8'))
FACTOID_SINGLE_ANSWER_INSTRUCTION = _prompt_registry['prompts'][SINGLE_ANSWER_PROMPT_ID]['instructions']['factoid']

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Project root:', PROJECT_ROOT)
print('Device:', torch.cuda.get_device_name(DEVICE) if DEVICE.type == 'cuda' else 'CPU (OpenAI-only evaluation is supported)')

from cse_dpo.factoid_recovery import (
    RECOVERY_PROMPT_ID, RECOVERY_SYSTEM_PROMPT,
    recover_factoid_answers, score_factoid_rows_with_java, verify_recovery_with_java,
)


In [ ]:
# Select one or more entries below. Add a local adapter path to MODEL_VARIANTS for new runs.
MODEL_VARIANTS = {
    # 3B strict-extractive SFT: best all-dev and supported-dev MRR at step 50.
    'evidence_per_alias_sft_dropout_strict_extractive_3b_best_mrr': PROJECT_ROOT / 'Artifacts/Factoid_SFT/models/evidence_grounded_per_supported_alias_qwen25_3b_lora_dropout_005_strict_extractive/adapter_best_evidence_mrr',
    # HEA-DPO: historical dev MRR 0.4625 at epoch 2, step 10.
    'hea_dpo_strict_extractive_historical_eval_best_mrr': PROJECT_ROOT / 'Artifacts/cse_dpo/hea_dpo_qwen25_05b/runs/hea_dpo_strict_extractive_best_mrr_historical_eval_20260909_063641_381119_seed3407/best_model_by_dev_mrr',
    'full_sft': PROJECT_ROOT / 'Artifacts/Factoid_SFT/models/full_resources_single_answer_qwen25_05b/adapter',
    'full_per_alias_sft_dropout': PROJECT_ROOT / 'Artifacts/Factoid_SFT/models/full_resources_per_gold_alias_qwen25_05b_lora_dropout_005/adapter',
    'evidence_sft': PROJECT_ROOT / 'Artifacts/Factoid_SFT/models/evidence_grounded_single_answer_qwen25_05b/adapter',
    'evidence_per_alias_sft_dropout': PROJECT_ROOT / 'Artifacts/Factoid_SFT/models/evidence_grounded_per_supported_alias_qwen25_05b_lora_dropout_005/adapter',
    'evidence_per_alias_sft_dropout_strict_extractive': PROJECT_ROOT / 'Artifacts/Factoid_SFT/models/evidence_grounded_per_supported_alias_qwen25_05b_lora_dropout_005_strict_extractive/adapter',
    'evidence_per_alias_sft_dropout_literal_copy_sft': PROJECT_ROOT / 'Artifacts/Factoid_SFT/models/evidence_grounded_per_supported_alias_qwen25_05b_lora_dropout_005_literal_copy/adapter',
    # Standard DPO trained from evidence_per_alias_sft_dropout on evidence_all_supported_local_only pairs.
    'standard_dpo_evidence_per_alias_all_supported_best_loss': PROJECT_ROOT / 'Artifacts/cse_dpo/pairwise_dpo_qwen25_05b/runs/standard_dpo_qwen25_05b_evidence_all_supported_local_only_run1_ctx4096_model_evidence_per_alias_sft_dropout_evidence_all_supported_evidence_per_alias_sft_dropout_standard_dpo_seed3407_run1_dropout_005_wd_002/best_model_by_loss',
    # This run's best generated-dev-MRR checkpoint is step zero, i.e. the unchanged source SFT.
    'standard_dpo_evidence_per_alias_all_supported_best_mrr_step_zero': PROJECT_ROOT / 'Artifacts/cse_dpo/pairwise_dpo_qwen25_05b/runs/standard_dpo_qwen25_05b_evidence_all_supported_local_only_run1_ctx4096_model_evidence_per_alias_sft_dropout_evidence_all_supported_evidence_per_alias_sft_dropout_standard_dpo_seed3407_run1_dropout_005_wd_002/best_model_by_mrr',
    # Standard DPO from the strict-extractive SFT on the balanced literal span candidates.
    'standard_dpo_span_balanced_strict_extractive_best_loss': PROJECT_ROOT / 'Artifacts/cse_dpo/pairwise_dpo_qwen25_05b/runs/standard_dpo_qwen25_05b_evidence_span_balanced_review_run1_ctx4096_model_evidence_per_alias_sft_dropout_strict_extractive_evidence_span_balanced_review_evidence_per_alias_sft_dropout_strict_extractive_standard_dpo_seed3407_run1_dropout_005_wd_002/best_model_by_loss',
    'standard_dpo_span_balanced_strict_extractive_best_mrr': PROJECT_ROOT / 'Artifacts/cse_dpo/pairwise_dpo_qwen25_05b/runs/standard_dpo_qwen25_05b_evidence_span_balanced_review_run1_ctx4096_model_evidence_per_alias_sft_dropout_strict_extractive_evidence_span_balanced_review_evidence_per_alias_sft_dropout_strict_extractive_standard_dpo_seed3407_run1_dropout_005_wd_002/best_model_by_mrr',
    # Cal-DPO from the strict-extractive SFT on the balanced literal span candidates.
    'cal_dpo_span_balanced_strict_extractive_best_loss': PROJECT_ROOT / 'Artifacts/cse_dpo/pairwise_dpo_qwen25_05b/runs/cal_dpo_qwen25_05b_evidence_span_balanced_review_run1_ctx4096_model_evidence_per_alias_sft_dropout_strict_extractive_evidence_span_balanced_review_evidence_per_alias_sft_dropout_strict_extractive_cal_dpo_seed3407_run1/best_model_by_loss',
    'cal_dpo_span_balanced_strict_extractive_best_mrr': PROJECT_ROOT / 'Artifacts/cse_dpo/pairwise_dpo_qwen25_05b/runs/cal_dpo_qwen25_05b_evidence_span_balanced_review_run1_ctx4096_model_evidence_per_alias_sft_dropout_strict_extractive_evidence_span_balanced_review_evidence_per_alias_sft_dropout_strict_extractive_cal_dpo_seed3407_run1/best_model_by_mrr',
    # Cal-DPO on sampled span negatives, restricted to their gold-bearing snippets.
    'cal_dpo_candidate_bank_gold_snippet_compact_strict_extractive_best_mrr': PROJECT_ROOT / 'Artifacts/cse_dpo/pairwise_dpo_qwen25_05b/runs/cal_dpo_qwen25_05b_candidate_bank_gold_snippet_spans_compact_review_run1_ctx4096_model_evidence_per_alias_sft_dropout_strict_extractive_candidate_bank_gold_snippet_spans_compact_review_evidence_per_alias_sft_dropout_strict_extractive_cal_dpo_seed3407_run1/best_model_by_mrr',
    'standard_dpo_strict_best_mrr': PROJECT_ROOT / 'Artifacts/cse_dpo/pairwise_dpo_qwen25_05b/runs/standard_dpo_qwen25_05b_evidence_strict_local_run1_ctx4096_model_evidence_sft_evidence_strict_local_evidence_sft_standard_dpo_seed3407_run1/best_model_by_mrr',
    'cal_dpo_strict_best_mrr': PROJECT_ROOT / 'Artifacts/cse_dpo/pairwise_dpo_qwen25_05b/runs/cal_dpo_qwen25_05b_evidence_strict_local_run1_ctx4096_model_evidence_sft_evidence_cal_dpo_seed3407_run1/best_model_by_mrr',
    'softmax_dpo_strict_best_mrr': PROJECT_ROOT / 'Artifacts/cse_dpo/pairwise_dpo_qwen25_05b/runs/softmax_dpo_qwen25_05b_evidence_strict_local_run1_ctx4096_run_20260831_101630/best_model_by_mrr',
}
# Most models use the aligned prompt. The strict-extractive and literal-copy SFTs must be evaluated with their respective training prompts.
MODEL_PROMPT_IDS = {
    'evidence_per_alias_sft_dropout_strict_extractive_3b_best_mrr': 'factoid-single-answer-extractive-v1',
    'hea_dpo_strict_extractive_historical_eval_best_mrr': 'factoid-single-answer-extractive-v1',
    'standard_dpo_span_balanced_strict_extractive_best_loss': 'factoid-single-answer-extractive-v1',
    'standard_dpo_span_balanced_strict_extractive_best_mrr': 'factoid-single-answer-extractive-v1',
    'cal_dpo_span_balanced_strict_extractive_best_loss': 'factoid-single-answer-extractive-v1',
    'cal_dpo_span_balanced_strict_extractive_best_mrr': 'factoid-single-answer-extractive-v1',
    'cal_dpo_candidate_bank_gold_snippet_compact_strict_extractive_best_mrr': 'factoid-single-answer-extractive-v1',
    'evidence_per_alias_sft_dropout_strict_extractive': 'factoid-single-answer-extractive-v1',
    'evidence_per_alias_sft_dropout_literal_copy_sft': 'factoid-single-answer-literal-copy-v1',
}
SELECTED_MODELS = [
    'evidence_per_alias_sft_dropout_strict_extractive_3b_best_mrr',
    'hea_dpo_strict_extractive_historical_eval_best_mrr',
    # 'full_sft', 
    # 'full_per_alias_sft_dropout',
    # 'evidence_sft', 
    # 'evidence_per_alias_sft_dropout',
    # 'evidence_per_alias_sft_dropout_strict_extractive',
    # 'evidence_per_alias_sft_dropout_literal_copy_sft',
    # 'standard_dpo_evidence_per_alias_all_supported_best_loss',
    # 'standard_dpo_span_balanced_strict_extractive_best_loss',
    # 'standard_dpo_span_balanced_strict_extractive_best_mrr',
    # 'cal_dpo_span_balanced_strict_extractive_best_loss',
    # 'cal_dpo_span_balanced_strict_extractive_best_mrr',
    # 'cal_dpo_candidate_bank_gold_snippet_compact_strict_extractive_best_mrr',
    # 'standard_dpo_strict_best_mrr', 
    # # 'cal_dpo_strict_best_mrr', 
    # 'softmax_dpo_strict_best_mrr'
    ]

BASE_MODEL_REF = PROJECT_ROOT / 'models/Qwen2.5-0.5B-Instruct'
# Resolve each adapter against its own base, including the optional reviewer.
MODEL_BASE_REFS = {name: BASE_MODEL_REF for name in MODEL_VARIANTS}
MODEL_BASE_REFS['evidence_per_alias_sft_dropout_strict_extractive_3b_best_mrr'] = PROJECT_ROOT / 'models/Qwen2.5-3B-Instruct'

MAX_SEQ_LENGTH = 9000
MAX_NEW_TOKENS = 64
# Choose the evaluation data/context independently of SELECTED_MODELS.
# - 'full_test': all 95 factoids with all provided resources.
# - 'evidence_supported_full_context': 82 supported factoids with all provided resources.
# - 'evidence_supported_gold_snippets': 82 supported factoids with only gold-bearing [BS]...[ES] snippets.
EVALUATION_DATA_VARIANT = 'evidence_supported_gold_snippets'
assert EVALUATION_DATA_VARIANT in {'full_test', 'evidence_supported_full_context', 'evidence_supported_gold_snippets'}

# Local and OpenAI selections are independent. For OpenAI-only evaluation, set
# SELECTED_MODELS = [] and OPENAI_MODELS = ['gpt-5-mini'].
# OpenAI requests send the question and its provided snippets to the API.
OPENAI_MODELS = []
OPENAI_API_KEY_PATH = PROJECT_ROOT / 'open_ai_api.txt'
OPENAI_MAX_OUTPUT_TOKENS = 64
OPENAI_TEMPERATURE = 0.0  # Set to None for models that do not accept this parameter.
OPENAI_MAX_RETRIES = 3
# Optional rank-2 review. Rank 1 always remains the original selected answer.
ENABLE_SECOND_PASS_RECOVERY = True
SECOND_PASS_MODEL_NAME = 'evidence_per_alias_sft_dropout_strict_extractive'
SECOND_PASS_ADAPTER = MODEL_VARIANTS[SECOND_PASS_MODEL_NAME]
SECOND_PASS_BASE_MODEL_REF = MODEL_BASE_REFS[SECOND_PASS_MODEL_NAME]
SECOND_PASS_MAX_ATTEMPTS = 2  # Retry once after malformed/duplicate output.
SECOND_PASS_MAX_NEW_TOKENS = 64
if ENABLE_SECOND_PASS_RECOVERY and SELECTED_MODELS:
    assert (SECOND_PASS_ADAPTER / 'adapter_config.json').is_file()
    assert (SECOND_PASS_BASE_MODEL_REF / 'config.json').is_file()
    print('Rank-2 reviewer:', SECOND_PASS_MODEL_NAME)
    print('Recovery prompt:', RECOVERY_PROMPT_ID)
    print(RECOVERY_SYSTEM_PROMPT)
RECOVERY_SETTINGS = {
    'enabled': ENABLE_SECOND_PASS_RECOVERY,
    'model_name': SECOND_PASS_MODEL_NAME,
    'adapter_path': str(SECOND_PASS_ADAPTER),
    'base_model': str(SECOND_PASS_BASE_MODEL_REF),
    'prompt_id': RECOVERY_PROMPT_ID,
    'system_prompt': RECOVERY_SYSTEM_PROMPT,
    'max_attempts': SECOND_PASS_MAX_ATTEMPTS,
    'max_new_tokens': SECOND_PASS_MAX_NEW_TOKENS,
    'decoding': 'greedy',
    'preserve_rank1': True,
    'duplicate_policy': 'ignore case, whitespace and punctuation for duplicate checks only',
    'rejected_attempt_policy': 'keep original only after retries; never fabricate an answer',
    'evidence': 'same resources as the first pass; gold-filtered variants remain oracle diagnostics',
    'scoring': 'official BioASQ Java Phase-B scorer (top-5 ranked factoid evaluation)',
}

RUN_NAME = None  # Set a descriptive label, or leave None for a timestamped directory.

unknown_models = sorted(set(SELECTED_MODELS) - set(MODEL_VARIANTS))
assert not unknown_models, f'Unknown model names: {unknown_models}'
unknown_prompt_ids = sorted(set(MODEL_PROMPT_IDS) - set(MODEL_VARIANTS))
assert not unknown_prompt_ids, f'Prompt IDs configured for unknown models: {unknown_prompt_ids}'
for prompt_id in MODEL_PROMPT_IDS.values():
    assert prompt_id in _prompt_registry['prompts'], f'Unknown prompt ID: {prompt_id}'
for model_name in SELECTED_MODELS:
    adapter_path = MODEL_VARIANTS[model_name]
    assert (adapter_path / 'adapter_config.json').exists(), f'Missing adapter: {adapter_path}'
if SELECTED_MODELS:
    assert DEVICE.type == 'cuda', 'A CUDA GPU is required only when evaluating local adapters.'
    assert (BASE_MODEL_REF / 'config.json').exists(), f'Missing base model: {BASE_MODEL_REF}'
if OPENAI_MODELS:
    assert OPENAI_API_KEY_PATH.exists(), f'Missing OpenAI API key file: {OPENAI_API_KEY_PATH}'

for model_name in SELECTED_MODELS:
    model_base = MODEL_BASE_REFS[model_name]
    assert (model_base / 'config.json').is_file(), f'Missing base model: {model_base}'
model_size_tags = sorted({'3b' if MODEL_BASE_REFS[name].name == 'Qwen2.5-3B-Instruct' else '05b' for name in SELECTED_MODELS})
model_size_tag = '_'.join(model_size_tags) if model_size_tags else '05b'

run_suffix = RUN_NAME or datetime.now().strftime('%Y%m%d_%H%M%S')
if ENABLE_SECOND_PASS_RECOVERY and SELECTED_MODELS:
    run_suffix += '_rank2_recovery'
OUTPUT_DIR = PROJECT_ROOT / 'Artifacts/Factoid_SFT/evaluations' / f'factoid_test_qwen25_{model_size_tag}_{EVALUATION_DATA_VARIANT}_{run_suffix}'
if OUTPUT_DIR.exists():
    raise FileExistsError(f'Refusing to overwrite an existing evaluation: {OUTPUT_DIR}')
OUTPUT_DIR.mkdir(parents=True)
print('Selected local models:', SELECTED_MODELS)
print('Selected OpenAI models:', OPENAI_MODELS)
print('Output directory:', OUTPUT_DIR)


In [ ]:
TEST_DIR = PROJECT_ROOT / 'data/Task13BTest'
TEST_FILES = sorted(TEST_DIR.glob('*_golden.json'))
assert len(TEST_FILES) == 4, f'Expected four test batches, found: {TEST_FILES}'


def test_source_args():
    return type('SourceArgs', (), {
        'question_types': ['factoid'],
        'max_summary_answers': 5,
        'max_factoid_answers': 5,
        'max_list_items': 100,
        'max_resources': 0,  # Keep all snippets, as in the full-resource dev evaluation.
        'max_resource_chars': 0,
        'resource_selection': 'first',
        'resource_granularity': 'document',
        'resource_window_mode': 'single',
        'resource_reranker_model': None,
        'resource_reranker_article_model': None,
        'resource_reranker_device': None,
        'resource_reranker_batch_size': 1,
        'local_files_only': True,
        'limit': None,
    })()


test_examples = load_eval_examples(TEST_FILES, test_source_args(), {'factoid': FACTOID_SINGLE_ANSWER_INSTRUCTION})
test_gold_examples = load_gold_examples(TEST_FILES)
assert len(test_examples) == 95, f'Expected 95 factoid test questions, found {len(test_examples)}'


def flatten_aliases(value):
    if isinstance(value, str):
        return [value]
    if isinstance(value, list):
        return [alias for item in value for alias in flatten_aliases(item)]
    return []


def is_evidence_supported(question):
    evidence = ' '.join(
        snippet.get('text', '') for snippet in question.get('snippets', []) if isinstance(snippet, dict)
    )
    normalized_evidence = normalize_answer_surface(evidence)
    return any(
        normalized_alias and f' {normalized_alias} ' in f' {normalized_evidence} '
        for alias in flatten_aliases(question.get('exact_answer', []))
        if (normalized_alias := normalize_answer_surface(alias))
    )


question_batch = {}
evidence_supported_ids = set()
for test_file in TEST_FILES:
    for question in json.loads(test_file.read_text(encoding='utf-8'))['questions']:
        if question.get('type') == 'factoid':
            question_batch[question['id']] = test_file.stem
            if is_evidence_supported(question):
                evidence_supported_ids.add(question['id'])

raw_questions={
    question['id']:question
    for test_file in TEST_FILES
    for question in json.loads(test_file.read_text(encoding='utf-8'))['questions']
    if question.get('type')=='factoid'
}

def gold_bearing_snippets(question):
    aliases=flatten_aliases(question.get('exact_answer', []))
    snippets=[]
    for snippet in question.get('snippets', []):
        if not isinstance(snippet, dict):
            continue
        text=snippet.get('text', '').strip()
        normalized_text=normalize_answer_surface(text)
        if text and any((normalized_alias:=normalize_answer_surface(alias)) and f' {normalized_alias} ' in f' {normalized_text} ' for alias in aliases):
            snippets.append(f'[BS]{text}[ES]')
    return tuple(snippets)

evidence_full_context_examples=[example for example in test_examples if example.question_id in evidence_supported_ids]
gold_snippet_examples=[]
for example in evidence_full_context_examples:
    snippets=gold_bearing_snippets(raw_questions[example.question_id])
    assert snippets, f'Missing gold-bearing snippet for {example.question_id}'
    gold_snippet_examples.append(EvalExample(
        question_id=example.question_id, question_type=example.question_type, body=example.body,
        instruction=example.instruction, resources=snippets, gold_output=example.gold_output,
        source_path=example.source_path, raw_question=example.raw_question,
    ))
assert len(evidence_full_context_examples)==82
assert len(gold_snippet_examples)==82
evaluation_examples={
    'full_test':test_examples,
    'evidence_supported_full_context':evidence_full_context_examples,
    'evidence_supported_gold_snippets':gold_snippet_examples,
}[EVALUATION_DATA_VARIANT]

def evaluation_examples_for_prompt(prompt_id):
    instruction = _prompt_registry['prompts'][prompt_id]['instructions']['factoid']
    return [EvalExample(
        question_id=example.question_id, question_type=example.question_type, body=example.body,
        instruction=instruction, resources=example.resources, gold_output=example.gold_output,
        source_path=example.source_path, raw_question=example.raw_question,
    ) for example in evaluation_examples]

print('Test batches:', [path.name for path in TEST_FILES])
print(f'Factoid test questions: {len(test_examples)}')
print(f'Evidence-supported test questions: {len(evidence_supported_ids)}')
print(f'Evaluation data variant: {EVALUATION_DATA_VARIANT}; questions: {len(evaluation_examples)}')


In [ ]:
def release_model(model=None):
    if model is not None:
        del model
    gc.collect()
    torch.cuda.empty_cache()


def load_model_and_tokenizer(adapter_path: Path, *, base_model_ref=None):
    base_model_ref = BASE_MODEL_REF if base_model_ref is None else Path(base_model_ref)
    tokenizer_path = adapter_path if (adapter_path / 'tokenizer_config.json').exists() else base_model_ref
    tokenizer = AutoTokenizer.from_pretrained(str(tokenizer_path), local_files_only=True)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = 'right'

    quantization = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )
    base_model = AutoModelForCausalLM.from_pretrained(
        str(base_model_ref),
        quantization_config=quantization,
        device_map={'': DEVICE.index or 0},
        low_cpu_mem_usage=True,
        local_files_only=True,
    )
    model = PeftModel.from_pretrained(base_model, str(adapter_path), local_files_only=True)
    model.config.use_cache = False
    model.eval()
    return model, tokenizer



EXPECTED_EVALUATION_PROMPTS = {
    'evidence_per_alias_sft_dropout_strict_extractive_3b_best_mrr': 'factoid-single-answer-extractive-v1',
    'hea_dpo_strict_extractive_historical_eval_best_mrr': 'factoid-single-answer-extractive-v1',
    'standard_dpo_span_balanced_strict_extractive_best_loss': 'factoid-single-answer-extractive-v1',
    'standard_dpo_span_balanced_strict_extractive_best_mrr': 'factoid-single-answer-extractive-v1',
    'cal_dpo_span_balanced_strict_extractive_best_loss': 'factoid-single-answer-extractive-v1',
    'cal_dpo_span_balanced_strict_extractive_best_mrr': 'factoid-single-answer-extractive-v1',
    'cal_dpo_candidate_bank_gold_snippet_compact_strict_extractive_best_mrr': 'factoid-single-answer-extractive-v1',
    'evidence_per_alias_sft_dropout_strict_extractive': 'factoid-single-answer-extractive-v1',
    'evidence_per_alias_sft_dropout_literal_copy_sft': 'factoid-single-answer-literal-copy-v1',
}
for model_name in SELECTED_MODELS:
    expected_prompt_id = EXPECTED_EVALUATION_PROMPTS.get(model_name)
    if expected_prompt_id is not None:
        actual_prompt_id = MODEL_PROMPT_IDS.get(model_name, SINGLE_ANSWER_PROMPT_ID)
        assert actual_prompt_id == expected_prompt_id, (
            f'{model_name} must use {expected_prompt_id}, but received {actual_prompt_id}. '
            'Restart the kernel and run the configuration cell before evaluating.'
        )

all_summaries = []
for model_name in SELECTED_MODELS:
    adapter_path = MODEL_VARIANTS[model_name]
    model_prompt_id = MODEL_PROMPT_IDS.get(model_name, SINGLE_ANSWER_PROMPT_ID)
    model_evaluation_examples = evaluation_examples_for_prompt(model_prompt_id)
    model_dir = OUTPUT_DIR / model_name
    model_dir.mkdir()
    print(f'\nEvaluating {model_name}: {adapter_path} (prompt: {model_prompt_id})')
    model, tokenizer = load_model_and_tokenizer(adapter_path, base_model_ref=MODEL_BASE_REFS[model_name])
    try:
        prompt_rows = []
        evaluation_input_records = []
        for example in model_evaluation_examples:
            rendered_prompt = render_prompt(tokenizer, example, chat_template='qwen-2.5', prompt_format='chat')
            prompt_rows.append({'question_id': example.question_id, 'prompt': rendered_prompt})
            evaluation_input_records.append({
                'question_id': example.question_id,
                'question_type': example.question_type,
                'prompt_id': model_prompt_id,
                'question': example.body,
                'resources': list(example.resources),
                'resource_count': len(example.resources),
                'gold_output': example.gold_output,
                'rendered_prompt': rendered_prompt,
            })
        generated_rows, missing_ids = build_generated_eval_rows(prompt_rows, test_gold_examples)
        assert not missing_ids, f'Missing gold records for: {missing_ids[:5]}'
        summary, metrics = evaluate_generated_bioasq(
            model, tokenizer, generated_rows,
            max_seq_length=MAX_SEQ_LENGTH,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            progress_factory=tqdm,
            progress_desc=f'Test generation: {model_name}',
            official_output_dir=model_dir / 'first_pass_official',
            official_model_label=model_name,
        )
    finally:
        del model, tokenizer
        release_model()

    # Free the first model before loading the reviewer, including on a small GPU.
    first_pass_metrics = [dict(row) for row in metrics]
    recovery_records = []
    recovery_summary = {}
    if ENABLE_SECOND_PASS_RECOVERY:
        reviewer, reviewer_tokenizer = load_model_and_tokenizer(
            SECOND_PASS_ADAPTER, base_model_ref=SECOND_PASS_BASE_MODEL_REF,
        )
        try:
            recovery_summary, metrics, recovery_records = recover_factoid_answers(
                reviewer, reviewer_tokenizer, metrics, evaluation_input_records, test_gold_examples,
                max_seq_length=MAX_SEQ_LENGTH,
                max_new_tokens=SECOND_PASS_MAX_NEW_TOKENS,
                max_attempts=SECOND_PASS_MAX_ATTEMPTS,
                progress_factory=tqdm,
                official_output_dir=model_dir / 'recovery_official',
                project_root=PROJECT_ROOT,
            )
            summary.update(recovery_summary)
        finally:
            del reviewer, reviewer_tokenizer
            release_model()


    for row in metrics:
        row['model_name'] = model_name
        row['batch'] = question_batch.get(row['question_id'])
        row['is_evidence_supported'] = row['question_id'] in evidence_supported_ids

    supported_metrics = [row for row in metrics if row['is_evidence_supported']]
    unsupported_metrics = [row for row in metrics if not row['is_evidence_supported']]
    supported_official = score_factoid_rows_with_java(
        supported_metrics, test_gold_examples, model_dir / 'supported_subset_official', PROJECT_ROOT,
        model_label=f'{model_name}-supported-subset') if supported_metrics else None
    unsupported_official = score_factoid_rows_with_java(
        unsupported_metrics, test_gold_examples, model_dir / 'unsupported_subset_official', PROJECT_ROOT,
        model_label=f'{model_name}-unsupported-subset') if unsupported_metrics else None
    if ENABLE_SECOND_PASS_RECOVERY:
        (model_dir / 'first_pass_metrics.json').write_text(
            json.dumps(first_pass_metrics, ensure_ascii=False, indent=2), encoding='utf-8',
        )
        (model_dir / 'second_pass_recovery.jsonl').write_text(
            ''.join(json.dumps(row, ensure_ascii=False) + '\n' for row in recovery_records), encoding='utf-8',
        )
        (model_dir / 'recovery_settings.json').write_text(
            json.dumps(RECOVERY_SETTINGS, indent=2), encoding='utf-8',
        )
    (model_dir / 'evaluation_inputs.jsonl').write_text(
        ''.join(json.dumps(record, ensure_ascii=False) + '\n' for record in evaluation_input_records),
        encoding='utf-8',
    )
    pd.DataFrame(metrics).to_csv(model_dir / 'per_question_metrics.csv', index=False)
    (model_dir / 'per_question_metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')

    if ENABLE_SECOND_PASS_RECOVERY:
        verify_recovery_with_java(metrics, test_gold_examples, model_dir, PROJECT_ROOT)

    record = {
        **recovery_summary,
        'second_pass_recovery': ENABLE_SECOND_PASS_RECOVERY,
        'model_name': model_name,
        'scoring_backend': summary['scoring_backend'],
        'adapter_path': str(adapter_path),
        'base_model_path': str(MODEL_BASE_REFS[model_name]),
        'prompt_id': model_prompt_id,
        'test_question_count': summary['question_count'],
        'test_mrr': summary['mrr'],
        'test_strict_accuracy': summary['strict_accuracy'],
        'test_lenient_accuracy': summary['lenient_accuracy'],
        'evidence_supported_count': len(supported_metrics),
        'evidence_supported_mrr': supported_official['mrr'] if supported_official else None,
        'unsupported_count': len(unsupported_metrics),
        'unsupported_mrr': unsupported_official['mrr'] if unsupported_official else None,
    }
    (model_dir / 'summary.json').write_text(json.dumps(record, indent=2), encoding='utf-8')
    all_summaries.append(record)

comparison = pd.DataFrame(all_summaries)
if not comparison.empty:
    comparison = comparison.sort_values('test_mrr', ascending=False)
comparison.to_csv(OUTPUT_DIR / 'model_comparison.csv', index=False)
(OUTPUT_DIR / 'manifest.json').write_text(json.dumps({
    'model_base_refs': {name: str(MODEL_BASE_REFS[name]) for name in SELECTED_MODELS},
    'second_pass_recovery': RECOVERY_SETTINGS,
    'test_files': [str(path) for path in TEST_FILES],
    'selected_models': {name: str(MODEL_VARIANTS[name]) for name in SELECTED_MODELS},
    'max_seq_length': MAX_SEQ_LENGTH,
    'max_new_tokens': MAX_NEW_TOKENS,
    'model_prompt_ids': {name: MODEL_PROMPT_IDS.get(name, SINGLE_ANSWER_PROMPT_ID) for name in SELECTED_MODELS},
    'evaluation_data_variant': EVALUATION_DATA_VARIANT,
    'evaluation_question_count': len(evaluation_examples),
    'evaluation_inputs_artifact': '<model_name>/evaluation_inputs.jsonl contains the exact resources and rendered prompt for every generated prediction',
    'decoding': 'greedy',
    'evidence_supported_definition': 'at least one normalized official gold alias occurs in the question snippets',
}, indent=2), encoding='utf-8')
comparison
